# Burn severity workflow
version 8

this notebook takes a file containing one or more burn extent polygons and perfoms the burn severity workflow on them. the out put is either single files per polygon or if desiered one merged geojson for all input polygons

note: combining the results of more that roughly 50 fires into one geojson results in a very large file that is more or less unuseable with a desktop computer and this is not adviseable. 

In [1]:
import datacube
import geopandas as gpd
from datacube.utils.cog import write_cog
import os

from burn_severity_notebook_helper import *

In [6]:
dc = datacube.Datacube(app="Burnt_severity")

## Before we run the code set up out-put locations and file formats

define the folder location where the output polygons will be saved

for locations on your scratch space you don't need to include '/home/jovyan/' in the address

In [7]:
output_folder = '/home/jovyan/gdata1/projects/Hazards/burn_severity/results'
#change this path if you are saving in a different location.

file_format = 'geojson'
#be default the files are saved as geojsons

merge_into_one_file = True
#change this to True if you want to combine all outputs into one file

## Define input file

this is the file containing the burn extent polygons you wish to perform burn severity mapping on. Ic can be located in the 'gdata1' share drive or on your own scratch space. by default it is pointing at the 'early cut' of polygons provided by Aurora. 

Any (vector) file format will work, Esri Shapefile, geojson, json, geopackage

(if you are using an esri shapefile they come in four seperate file parts; .cpg, .dbf, .prj, .shp. all parts need to sit in the same folder but you point the script at the '.shp' part )


In [22]:
input_file = '/home/jovyan/gdata1/projects/Hazards/burn_severity/fire_extents_2025-26_summer/all_fires_at_29JAN26.gpkg'
#define input file

poly = gpd.read_file(input_file)
#open as a geopandas geodataframe and have a look at the first few rows

#let's peek at the first one
poly.iloc[0]

fire_id                                                   102636004
fire_name                                   Tallandoon - Yabba Road
fire_type                                        Current Burnt Area
ignition_date                                   2025-11-01 00:00:00
capt_date                                       2025-11-03 00:00:00
capt_method                              Satellite Imagery Sentinel
area_ha                                                        None
perim_km                                                       None
state                                                           VIC
agency            Vic Department of Energy, Environment and Clim...
date_retrieved                                  2025-11-04 00:00:00
date_processed                                                 None
geometry          MULTIPOLYGON (((147.250514901 -36.422828973, 1...
Name: 0, dtype: object

## check attributes

code asumes your input polygons have the attributes defined by the trigger product data dictionary. Shapefiles do this fun thing where they shorten long attribute names, so if you are using a shapefile we will fix that up now. 
        'fire_id',
        'fire_name',
        'fire_type',
        'ignition_date',
        'capt_date',
        'capt_method',
        'area_ha',
        'perim_km',
        'state',
        'agency'
        'date_retrieved'
        'date_processed'

if your polygon dosn't have the correct attibutes (they can be empty they just need to be there) the script will get Angry at you and not run

        

In [16]:
#test attribute format

poly = test_polygon_attributes(poly)

attibutes are long, no need to change


## pre-filtering

pre-filtering steps on polygons:
- remove any polygons with no area or smaller than 1 ha
- check for duplicates for the same fire and perfrom spatial join on them


In [12]:
#filter on size 
poly = poly.drop(poly[poly.area_ha < 1].index)

#list all fire 'id' names
list_of_fires = list(set(poly['fire_id']))

#perform spatial dissolve on overlapping polygons with same id
dissolved_duplicates = perform_spatial_dissolve(poly, list_of_fires)

/tmp/ipykernel_298/2845258603.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  poly['area_ha'] = round((poly['geometry'].area)/10000, 2)


ValueError: No objects to concatenate

In [13]:
list_of_fires

[]

In [ ]:

dissolved_duplicates.iloc[0]

## map burn severity


In [ ]:

#this essentially creates a numerical list of each dissolved fire
for fires in range(0,len(dissolved_duplicates.index)):

    
    #select each polygon one-by -one
    fire_polygon = gpd.GeoDataFrame(dissolved_duplicates.iloc[fires].to_frame().T, geometry='geometry', crs=dissolved_duplicates.crs)

    #make a sting which is fire id and position in gpd to not over-write re-use of ids in QLD
    fire_id_forsave = f'{fire_polygon.fire_id.iloc[0]}_{fires}'
    
    #run burn severity mapping
    Severity_polygons, debug_layer = map_burn_severity(fire_polygon)

       # #save_debug layer to file
    write_cog(debug_layer.compute(), f'{output_folder}/DEA_burn_severity_debug_{fire_id_forsave}.tif',
             overwrite=True)

    Severity_polygons.to_file(f'{output_folder}/DEA_burn_severity_{fire_id_forsave}.{file_format}')
    
    print( f'I finished processing number {fires}')
    


## Run optional merge of geoploygons
if selected this will save the merged polygons in the Same Folder as the individual ones but with the name 'DEA_burn_severity_MERGED'

In [ ]:
if merge_into_one_file == True:
    
    # List all GeoJSON files in the directory
    geojson_files = [f for f in os.listdir(output_folder) if f.endswith(file_format)]
    
    
    # Read and concatenate all GeoDataFrames
    gdfs = []
    for file in geojson_files:
        filepath = os.path.join(output_folder, file)
        gdf = gpd.read_file(filepath)
        gdfs.append(gdf)
    
    # Merge into one GeoDataFrame
    merged_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    
    # # Save to one big GeoJSON
    merged_gdf.to_file(f"{output_folder}/DEA_burn_severity_MERGED.{file_format}")
 